# Baseline Pipeline: Fraud Detection (Leak-Free Benchmark)

> **Mục tiêu:** Xây dựng quy trình chuẩn sạch (Clean Pipeline) làm mốc tham chiếu cho 8 kịch bản Data Leakage.
>
> **Nguyên tắc chống rò rỉ:**
> 1. Tách tập kiểm thử độc lập (`Clean Test Set` - 20%) ngay từ đầu bằng `StratifiedSplit`.
> 2. Mọi bước tiền xử lý (Imputer, Scaler, Encoder) chỉ được `fit()` trên tập Train.
> 3. Đánh giá trên bộ chỉ số chuẩn: **PR-AUC (quan trọng nhất)**, ROC-AUC, F1-Score, Precision, Recall.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# =============================================================================
# 1. ĐƯỜNG DẪN DỮ LIỆU ĐỘNG (Chạy được trên mọi máy khi push Git)
# =============================================================================
current_dir = Path.cwd().resolve()
project_root = current_dir
while not (project_root / "data").exists() and project_root.parent != project_root:
    project_root = project_root.parent

DATA_PATH = project_root / "data" / "fraud_detection.csv"
OUTPUT_DIR = project_root / "model"
RANDOM_STATE = 42

print("Project Root:", project_root)
print("Data Path:   ", DATA_PATH)
print("Tồn tại file:", DATA_PATH.exists())

## 1. Đọc dữ liệu & Phân loại thuộc tính (Features & Target)

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")

TARGET = "Fraudulent"
DROP_COLS = ["Transaction_ID", "User_ID", TARGET]

NUM_COLS = [
    "Transaction_Amount",
    "Time_of_Transaction",
    "Previous_Fraudulent_Transactions",
    "Account_Age",
    "Number_of_Transactions_Last_24H"
]

CAT_COLS = [
    "Transaction_Type",
    "Device_Used",
    "Location",
    "Payment_Method"
]

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

print(f"Tỷ lệ nhãn gian lận: {y.mean():.4f} ({y.sum()} / {len(y)})")

## 2. Tách Clean Test Set (Clean Holdout - 20%)

> **Nguyên tắc chống rò rỉ:** Tách tập kiểm thử độc lập ngay từ đầu bằng `stratify=y` trước mọi bước fit tiền xử lý.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Tập Train: {X_train.shape[0]} dòng | Tỷ lệ fraud: {y_train.mean():.4f} ({y_train.sum()} samples)")
print(f"Tập Test:  {X_test.shape[0]} dòng | Tỷ lệ fraud: {y_test.mean():.4f} ({y_test.sum()} samples)")

## 3. Xây dựng Tiền xử lý chuẩn chống rò rỉ (Chỉ fit trên Train)

In [ ]:
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, NUM_COLS),
        ("cat", cat_transformer, CAT_COLS)
    ]
)

## 4. Huấn luyện 3 mô hình Baseline & Đánh giá trên Clean Test

- **Logistic Regression** (Mô hình tuyến tính)
- **Decision Tree** (Cây quyết định)
- **Random Forest** (Mô hình ensemble)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=6,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

results = []
trained_pipelines = {}

for name, clf in models.items():
    print(f"Đang huấn luyện: {name}...")
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", clf)
    ])
    
    # Chỉ fit trên Train
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline
    
    # Đánh giá trên Clean Test
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    pr_auc = average_precision_score(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "PR-AUC": round(pr_auc, 4),
        "ROC-AUC": round(roc_auc, 4),
        "F1-Score": round(f1, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4)
    })

baseline_results_df = pd.DataFrame(results)
print("\n=== KẾT QUẢ BASELINE (CLEAN BENCHMARK) ===")
baseline_results_df

## 5. Lưu kết quả Baseline ra file làm mốc so sánh cho 8 kịch bản Leakage

In [ ]:
results_file = OUTPUT_DIR / "baseline_results.csv"
baseline_results_df.to_csv(results_file, index=False)
print(f"Đã lưu kết quả baseline vào: {results_file}")